# CodeAlong: Experiment Tracking and Model Registry with MLflow

`5. MLOps/2. End-to-End ML/heart_disease.ipynb` introduced MLflow with a few cells
logging a single logistic regression run. This CodeAlong goes further: track **several**
competing models on the same dataset, compare their runs, log artifacts alongside
metrics, and register the winning model so it can be reloaded and served without
needing the training code again.

See `project.txt` in this folder for the original DataCamp DataLab this CodeAlong is
based on.

## Why track experiments at all?

Without a tracking tool, "which hyperparameters gave the best model?" gets answered by
scrolling back through notebook output, or worse, by memory. MLflow logs every run's
parameters, metrics and artifacts to a queryable store, so the answer becomes a
one-line query instead of an archaeology exercise — the same problem
`6. Statistical Foundations for Data Science/10. Train-Test Split.ipynb` and
`11. Overfitting.ipynb` warn about when they say "report the CV mean and sd, not one
lucky number": tracking makes that discipline automatic instead of manual.

In [ ]:
import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score

SKF = StratifiedKFold(5, shuffle=True, random_state=0)

## 1. Point MLflow at a tracking store

`set_tracking_uri` tells MLflow where to write run data. A local folder is enough for
this CodeAlong (and matches what `heart_disease.ipynb` already uses, so both notebooks'
runs land in the same place); a real team would point this at a shared MLflow tracking
server instead.

In [ ]:
mlflow.set_tracking_uri("../../mlruns")
mlflow.set_experiment("Heart Disease - Model Comparison")

print("Tracking URI:", mlflow.get_tracking_uri())
print("Active experiment:", mlflow.get_experiment_by_name("Heart Disease - Model Comparison"))

## 2. The data

Same dataset as `heart_disease.ipynb`, so the two sets of runs are directly comparable
in the MLflow UI.

In [ ]:
df = pd.read_csv("../../5. MLOps/2. End-to-End ML/data/heart_disease_cleaned_2.csv", index_col=0)
X = df.drop(columns="target")
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42,
                                                     stratify=y)
print(f"train: {len(y_train)}, test: {len(y_test)}, prevalence: {y.mean():.3f}")

## 3. Log several competing models as separate runs

Each `with mlflow.start_run():` block is one experiment run. Inside it we log:

* **parameters** — the hyperparameters that define this run (`mlflow.log_param`)
* **metrics** — the numbers we care about (`mlflow.log_metric`)
* **the model itself** — as an artifact MLflow knows how to reload later (`mlflow.sklearn.log_model`)

In [ ]:
def train_and_log(name, model, params):
    with mlflow.start_run(run_name=name) as run:
        cv_auc = cross_val_score(model, X_train, y_train, cv=SKF, scoring="roc_auc").mean()
        model.fit(X_train, y_train)

        pred = model.predict(X_test)
        proba = model.predict_proba(X_test)[:, 1]
        test_acc = accuracy_score(y_test, pred)
        test_auc = roc_auc_score(y_test, proba)
        test_f1 = f1_score(y_test, pred)

        for k, v in params.items():
            mlflow.log_param(k, v)
        mlflow.log_metric("cv_roc_auc", cv_auc)
        mlflow.log_metric("test_accuracy", test_acc)
        mlflow.log_metric("test_roc_auc", test_auc)
        mlflow.log_metric("test_f1", test_f1)

        mlflow.sklearn.log_model(model, artifact_path="model")

        print(f"{name:<22} run_id={run.info.run_id}  cv_auc={cv_auc:.4f}  "
              f"test_auc={test_auc:.4f}  test_acc={test_acc:.4f}")
        return run.info.run_id, test_auc


runs = {}
runs["logistic"] = train_and_log(
    "logistic_regression",
    make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, C=1.0)),
    {"model_type": "LogisticRegression", "C": 1.0, "max_iter": 2000},
)
runs["tree"] = train_and_log(
    "decision_tree",
    DecisionTreeClassifier(max_depth=5, min_samples_leaf=5, random_state=0),
    {"model_type": "DecisionTree", "max_depth": 5, "min_samples_leaf": 5},
)
runs["forest_small"] = train_and_log(
    "random_forest_100",
    RandomForestClassifier(n_estimators=100, min_samples_leaf=3, random_state=0),
    {"model_type": "RandomForest", "n_estimators": 100, "min_samples_leaf": 3},
)
runs["forest_large"] = train_and_log(
    "random_forest_400",
    RandomForestClassifier(n_estimators=400, min_samples_leaf=3, random_state=0),
    {"model_type": "RandomForest", "n_estimators": 400, "min_samples_leaf": 3},
)

## 4. Log an artifact: a plot, not just numbers

Metrics are searchable and plottable in the MLflow UI directly, but sometimes you want
to save a specific figure (a confusion matrix, a feature-importance chart) alongside a
run. `log_figure` (or `log_artifact` for any file) attaches it to the run.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve

best_forest = RandomForestClassifier(n_estimators=400, min_samples_leaf=3, random_state=0)

with mlflow.start_run(run_name="random_forest_400_with_plot"):
    best_forest.fit(X_train, y_train)
    proba = best_forest.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, proba)

    fpr, tpr, _ = roc_curve(y_test, proba)
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.plot(fpr, tpr, color="steelblue", lw=2, label=f"AUC={auc:.3f}")
    ax.plot([0, 1], [0, 1], "k--", lw=1)
    ax.set_xlabel("false positive rate"); ax.set_ylabel("true positive rate")
    ax.set_title("ROC curve")
    ax.legend()

    mlflow.log_param("model_type", "RandomForest")
    mlflow.log_param("n_estimators", 400)
    mlflow.log_metric("test_roc_auc", auc)
    mlflow.log_figure(fig, "roc_curve.png")
    mlflow.sklearn.log_model(best_forest, artifact_path="model")
    plt.close(fig)

    print("Logged a run with an attached ROC curve artifact.")

## 5. Query and compare runs programmatically

The whole point of tracking: instead of eyeballing printed output, ask the tracking
store directly for the best run so far.

In [ ]:
experiment = mlflow.get_experiment_by_name("Heart Disease - Model Comparison")
results = mlflow.search_runs(experiment_ids=[experiment.experiment_id],
                             order_by=["metrics.test_roc_auc DESC"])

cols = ["run_id", "tags.mlflow.runName", "params.model_type", "metrics.cv_roc_auc",
       "metrics.test_roc_auc", "metrics.test_accuracy"]
available_cols = [c for c in cols if c in results.columns]
print(results[available_cols].round(4).to_string(index=False))

In [ ]:
best_run = results.iloc[0]
print(f"Best run by test ROC-AUC: {best_run['tags.mlflow.runName']}")
print(f"  run_id       : {best_run['run_id']}")
print(f"  test ROC-AUC : {best_run['metrics.test_roc_auc']:.4f}")

## 6. Reload a logged model — no training code needed

This is what makes tracked models actually useful in production: a downstream service
only needs the `run_id` (or a registered model name/version) to load the exact model
that produced a given run's metrics, without importing any of the training script.

In [ ]:
model_uri = f"runs:/{best_run['run_id']}/model"
loaded_model = mlflow.sklearn.load_model(model_uri)

# Confirm it reproduces the same predictions as the original in-memory model
reloaded_proba = loaded_model.predict_proba(X_test)[:, 1]
reloaded_auc = roc_auc_score(y_test, reloaded_proba)
print(f"Reloaded model test ROC-AUC: {reloaded_auc:.4f}")
print(f"Matches the logged metric  : {np.isclose(reloaded_auc, best_run['metrics.test_roc_auc'])}")

## 7. Register the winning model

The **model registry** is one level above individual runs: a named, versioned model
("HeartDiseaseClassifier" v1, v2, v3, ...) that a serving system can reference by name
and stage (`Staging`, `Production`) instead of by a specific run ID that changes every
time someone retrains.

In [ ]:
registered_name = "HeartDiseaseClassifier"
result = mlflow.register_model(model_uri=model_uri, name=registered_name)

print(f"Registered '{registered_name}' as version {result.version}")

client = mlflow.tracking.MlflowClient()
versions = client.search_model_versions(f"name='{registered_name}'")
for v in versions:
    print(f"  version {v.version}: run_id={v.run_id}, status={v.status}")

## What to try next

* Re-run the comparison cell with different hyperparameter grids and watch the leader
  board in `mlflow.search_runs` change — this is the natural next step toward the
  `GridSearchCV`/`RandomizedSearchCV` sweeps from
  `7. Machine Learning Fundamentals and Predictive Analytics/6. Random Forest.ipynb`,
  logging every candidate as its own run instead of only keeping the best one.
* Launch the MLflow UI locally (`mlflow ui --backend-store-uri ../../mlruns`) and browse
  the runs visually — sortable tables and parallel-coordinates plots across
  hyperparameters are much easier to read there than in a DataFrame.
* Promote the registered model to a `Production` stage
  (`client.transition_model_version_stage`) once you are satisfied with it, and load it
  in a separate serving script by `models:/HeartDiseaseClassifier/Production` instead of
  a specific run ID — that indirection is what lets you retrain and redeploy without
  changing the serving code at all.